# Google Trends revisited: the API - Lane B (worked solution)

**SISMID 2026 - Day 2, 9:00.** Before we add new streams, we fix the one you already know.

Yesterday you scraped Google Trends with `pytrends` and met its two problems: **HTTP 429**
rate limits, and values that **change between pulls** because the public endpoint is
sampled. Today we use the **Google Trends API** instead. It returns the **same normalized
0-100 index**, so nothing about your analysis changes, but it is reliable and
reproducible.

> Captured example of what a coding agent (Codex, Claude Code, or Antigravity CLI)
> produces from the **Lane A** prompts.


## Step 0: unlock the key, then the helper

In the **terminal**, before starting Jupyter:

```bash
source scripts/unlock-gt-api-key.sh     # asks for the Google Trends passcode
```

That exports `GT_API`. If Jupyter was already running, restart the kernel. Without a key,
every cell below falls back to a cached snapshot, so you are never blocked.

**Quota: 1,000 requests/day for the whole room, health topics only.**


In [ ]:
# --- Produced by the agent from the Plan A / Step 0 prompt ---
import pandas as pd, matplotlib.pyplot as plt, os, json, time
import urllib.request, urllib.parse, urllib.error

API_KEY = os.getenv('GT_API', '')
print('GT_API loaded' if API_KEY else 'No GT_API set -> the cached snapshots will be used')

ENDPOINT = 'https://www.googleapis.com/trends/v1beta/graph'
DATA = '../data/'

def _norm(c):
    return c.strip().replace(' ', '_').replace('/', '_').lstrip('_')

def gt_api_fetch(terms, geo='US', start='2021-07', end=None, key=None):
    """Google Trends API (getGraph): the SAME normalized 0-100 index pytrends
    returns, but from a documented API. No 429s, and identical values every pull.
    Dates are YYYY-MM; Google picks weekly vs monthly from the window length."""
    end = end or time.strftime('%Y-%m')
    key = key or API_KEY
    if not key:
        raise RuntimeError("GT_API not set (source scripts/unlock-gt-api-key.sh)")
    params = [('key', key)] + [('terms', t) for t in terms] + [
        ('restrictions.geo', geo), ('restrictions.startDate', start),
        ('restrictions.endDate', end)]
    url = ENDPOINT + '?' + urllib.parse.urlencode(params)
    try:
        with urllib.request.urlopen(url, timeout=45) as r:
            lines = json.loads(r.read()).get('lines') or []
    except urllib.error.HTTPError as e:
        raise RuntimeError(f'HTTP {e.code}: {e.read().decode("utf-8","replace")[:200]}') from e
    if not lines:
        raise RuntimeError(f'empty response for {terms} in {geo}')
    frames = [pd.DataFrame({'date': pd.to_datetime([p['date'] for p in l['points']]),
                            _norm(l['term']): [p['value'] for p in l['points']]}).set_index('date')
              for l in lines]
    return pd.concat(frames, axis=1).reset_index().sort_values('date').reset_index(drop=True)


## Step 1: the same dengue signal, from the API

Yesterday's topic (dengue in Mexico), pulled the reliable way.


In [ ]:
# --- Produced by the agent from the Plan A / Step 1 prompt ---
TERMS, GEO = ['dengue', 'sintomas de dengue', 'mosquito'], 'MX'
try:
    dengue = gt_api_fetch(TERMS, geo=GEO, start='2021-07')
    src = 'live API'
except Exception as e:
    print('API unavailable:', e)
    dengue = pd.read_csv(DATA + 'gt_api_dengue_mx_cached.csv', parse_dates=['date'])
    src = 'cached snapshot'
cols = [c for c in dengue.columns if c != 'date']
gap = dengue['date'].diff().dt.days.median()
print(f"{src}: {len(dengue)} points ({'weekly' if gap<=10 else 'monthly'}), "
      f"{dengue['date'].min().date()} -> {dengue['date'].max().date()}")
plt.figure(figsize=(11,4))
for c in cols: plt.plot(dengue['date'], dengue[c], label=c)
plt.legend(); plt.title('Google Trends API: dengue in Mexico')
plt.ylabel('interest (0-100)'); plt.tight_layout(); plt.show()


## Step 2: the difference that matters, reproducibility

`pytrends` reads a **sampled** endpoint: pull the same series twice and the numbers move.
The API is deterministic. That is what makes an analysis you can hand to someone else.


In [ ]:
# --- Produced by the agent from the Plan A / Step 2 prompt ---
try:
    a = gt_api_fetch(['dengue'], geo='MX', start='2024-01')
    b = gt_api_fetch(['dengue'], geo='MX', start='2024-01')
    print('two API pulls identical?', a['dengue'].tolist() == b['dengue'].tolist())
    print('  (with pytrends, this is normally False, and often a 429 instead)')
except Exception as e:
    print('Needs a key for the live check:', e)


## Step 3: term vs topic, the language lesson

This is the part that matters most for Day 2, because today we go multi-country.

A **term** is the literal string you typed. A **topic** is Google's Knowledge Graph entity
(`/m/0cycc` = Influenza) that folds together every spelling, synonym and **language** for
the concept. The API takes either.

Watch what happens to the English term `flu` outside the United States.


In [ ]:
# --- Produced by the agent from the Plan A / Step 3 prompt ---
try:
    frames = []
    for geo, label in [('US','United States'), ('FR','France'), ('IT','Italy'), ('MX','Mexico')]:
        c = gt_api_fetch(['flu', '/m/0cycc'], geo=geo, start='2022-01')
        c.columns = ['date', 'term_flu', 'topic_influenza']; c['geo'] = label
        frames.append(c)
    tvt = pd.concat(frames, ignore_index=True)
except Exception as e:
    print('API unavailable:', e, '-> cache')
    tvt = pd.read_csv(DATA + 'gt_api_flu_term_vs_topic_cached.csv', parse_dates=['date'])

summary = tvt.groupby('geo').agg(term_max=('term_flu','max'), term_mean=('term_flu','mean'),
                                 topic_max=('topic_influenza','max'),
                                 topic_mean=('topic_influenza','mean')).round(1)
display(summary)
fig, axes = plt.subplots(2, 2, figsize=(12,6), sharex=True)
for ax, (geo, g) in zip(axes.ravel(), tvt.groupby('geo')):
    ax.plot(g['date'], g['term_flu'], label='term "flu"')
    ax.plot(g['date'], g['topic_influenza'], label='topic /m/0cycc')
    ax.set_title(geo, fontsize=10); ax.set_ylim(0, 105)
axes[0,0].legend(fontsize=8)
plt.suptitle('An English term is nearly silent abroad; the topic sees the season everywhere')
plt.tight_layout(); plt.show()


## Step 4: what this buys you

- **Same signal**, so yesterday's models and plots still apply.
- **No 429s**, so a room of thirty can all pull at once.
- **Reproducible**, so your numbers do not drift between runs.
- **Topics**, so a single query works across languages, which is what makes the
  multi-country work later today possible.

Still true, and unchanged by the API: Google Trends measures **attention**, not infection,
and low-volume places return a lot of zeros. Those are properties of search itself.

**Now on to streams that fail differently:** Wikipedia pageviews (`01_...`) and wastewater
(`02_...`).


In [ ]:
out = 'gt_api_dengue_mx.csv'
dengue.to_csv(out, index=False)
print('saved', out, '|', len(dengue), 'rows,', list(dengue.columns))
